# Prediction API — Testing

> **Notebook flow:** 01 Setup → 02 EDA → 03 Train → 04 Docker → 05 Kubernetes → 06 Cleanup · 07 Azure Deploy · **[08 API Tests]**

This notebook tests the FastAPI prediction service using FastAPI's `TestClient` — no running server required.
It mirrors `03_ml_pipeline.ipynb` but focuses entirely on prediction code: `/health`, `/predict`, and `/predict/batch`.

**Prerequisite:** Run `03_ml_pipeline.ipynb §1` first — `artifacts/model.pkl` must exist before the API can load.

---

## What is tested

| Section | What it covers |
|---|---|
| §1 | Setup — load config, build sample payloads from CSV |
| §2 | `/health` endpoint — liveness/readiness probe |
| §3 | `/predict` — single-record scoring |
| §4 | `/predict` edge cases — empty features, auth enforcement |
| §5 | `/predict/batch` — multi-record scoring |
| §6 | `/predict/batch` edge cases — empty records, limit enforcement |
| §7 | Run `test_api.py` — full pytest suite for the API |
| §8 | Run `test_pipelines.py` — end-to-end pipeline tests |

## 1. Setup

Loads config, sets up the `TestClient`, and auto-generates sample payloads from the first row of the training CSV.
No feature names are hardcoded — works for any dataset configured in `config.yaml`.

In [ ]:
import sys
import os
import json
import yaml
import pandas as pd

ROOT = "/workspaces/marketing-model-mlops-azure"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# Set API key before importing the app so the dependency picks it up
TEST_API_KEY = "test-key-for-notebook"
os.environ["API_KEY"] = TEST_API_KEY
AUTH = {"X-API-Key": TEST_API_KEY}

# Load config — all paths come from here, nothing hardcoded below
with open(os.path.join(ROOT, "config.yaml")) as f:
    config = yaml.safe_load(f)

MODEL_PATH = os.path.join(ROOT, config["artifacts"]["model_path"])

print(f"Working directory: {os.getcwd()}")
print(f"Model path:        {MODEL_PATH}")
print(f"Model exists:      {os.path.exists(MODEL_PATH)}")

if not os.path.exists(MODEL_PATH):
    print()
    print("⚠️  model.pkl not found — run 03_ml_pipeline.ipynb §1 first.")

In [ ]:
# Auto-generate sample payloads from the first two rows of the training CSV.
# This avoids hardcoding dataset-specific column names.
_raw_path = os.path.join(ROOT, config["data"]["raw_path"])
_sep      = config["data"].get("separator", ",")
_target   = config["model"]["target_column"]

_rows = pd.read_csv(_raw_path, sep=_sep, nrows=5)
_feature_cols = [c for c in _rows.columns if c != _target]

def _to_py(v):
    """Convert numpy scalar → Python native type for JSON serialisation."""
    return v.item() if hasattr(v, "item") else v

def _row_to_features(df, idx):
    return {k: _to_py(v) for k, v in df[_feature_cols].iloc[idx].items()}

SINGLE_REQUEST = {"features": _row_to_features(_rows, 0)}
BATCH_REQUEST  = {"records": [_row_to_features(_rows, i) for i in range(3)]}

print(f"Feature columns ({len(_feature_cols)}): {_feature_cols}")
print(f"\nSINGLE_REQUEST keys: {list(SINGLE_REQUEST['features'].keys())}")
print(f"BATCH_REQUEST records: {len(BATCH_REQUEST['records'])}")

In [ ]:
# Start TestClient — this triggers the lifespan (model load) once
from fastapi.testclient import TestClient
from src.api.app import app

client = TestClient(app)
client.__enter__()
print("TestClient started — model loaded into app state.")

## 2. `/health` — Liveness Probe

Used by AKS readiness/liveness probes. Returns `200 healthy` when the model is loaded, `503 unhealthy` when it is not.

In [ ]:
r = client.get("/health")
print(f"Status: {r.status_code}")
print(f"Body:   {r.json()}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}"
assert r.json()["status"] == "healthy"
print("\n✅  /health OK")

In [ ]:
# Simulate model not loaded → expect 503
original_pipeline = app.state.pipeline
try:
    app.state.pipeline = None
    r = client.get("/health")
    print(f"Status: {r.status_code}")
    print(f"Body:   {r.json()}")
    assert r.status_code == 503
    assert r.json()["status"] == "unhealthy"
    print("\n✅  /health 503 when model not loaded — OK")
finally:
    app.state.pipeline = original_pipeline

## 3. `POST /predict` — Single-Record Scoring

Accepts `{"features": {col: value, ...}}`. Returns `{prediction, probability, label}`.

In [ ]:
r = client.post("/predict", json=SINGLE_REQUEST, headers=AUTH)
print(f"Status: {r.status_code}")
print(f"Body:   {json.dumps(r.json(), indent=2)}")

assert r.status_code == 200
data = r.json()
assert "prediction" in data
assert "probability" in data
assert "label" in data
assert data["prediction"] in (0, 1)
assert 0.0 <= data["probability"] <= 1.0
assert data["label"] in ("yes", "no")
expected_label = "yes" if data["prediction"] == 1 else "no"
assert data["label"] == expected_label, "label must match prediction"
print("\n✅  /predict single-record OK")

## 4. `POST /predict` — Edge Cases

In [ ]:
# Empty features dict → 422
r = client.post("/predict", json={"features": {}}, headers=AUTH)
print(f"Empty features → {r.status_code} (expected 422)")
assert r.status_code == 422

# Missing features key entirely → 422
r = client.post("/predict", json={}, headers=AUTH)
print(f"Missing features key → {r.status_code} (expected 422)")
assert r.status_code == 422

# No API key → 401
r = client.post("/predict", json=SINGLE_REQUEST)
print(f"No API key → {r.status_code} (expected 401)")
assert r.status_code == 401

# Wrong API key → 401
r = client.post("/predict", json=SINGLE_REQUEST, headers={"X-API-Key": "wrong"})
print(f"Wrong API key → {r.status_code} (expected 401)")
assert r.status_code == 401

print("\n✅  /predict edge cases OK")

## 5. `POST /predict/batch` — Multi-Record Scoring

Accepts `{"records": [{col: value, ...}, ...]}`. Returns `{"predictions": [{prediction, probability, label}, ...]}`. Maximum 1000 records per request.

In [ ]:
r = client.post("/predict/batch", json=BATCH_REQUEST, headers=AUTH)
print(f"Status: {r.status_code}")
data = r.json()
print(f"Predictions returned: {len(data['predictions'])} (expected {len(BATCH_REQUEST['records'])})")
print()
for i, pred in enumerate(data["predictions"]):
    print(f"  Record {i}: prediction={pred['prediction']}  probability={pred['probability']:.4f}  label={pred['label']}")

assert r.status_code == 200
assert len(data["predictions"]) == len(BATCH_REQUEST["records"])
for item in data["predictions"]:
    assert item["prediction"] in (0, 1)
    assert 0.0 <= item["probability"] <= 1.0
    assert item["label"] in ("yes", "no")
    expected_label = "yes" if item["prediction"] == 1 else "no"
    assert item["label"] == expected_label

print("\n✅  /predict/batch OK")

## 6. `POST /predict/batch` — Edge Cases

In [ ]:
# Empty records list → 422
r = client.post("/predict/batch", json={"records": []}, headers=AUTH)
print(f"Empty records → {r.status_code} (expected 422)")
assert r.status_code == 422

# > 1000 records → 422
big_batch = {"records": [SINGLE_REQUEST["features"]] * 1001}
r = client.post("/predict/batch", json=big_batch, headers=AUTH)
print(f"1001 records → {r.status_code} (expected 422)")
assert r.status_code == 422

# No API key → 401
r = client.post("/predict/batch", json=BATCH_REQUEST)
print(f"No API key → {r.status_code} (expected 401)")
assert r.status_code == 401

# Single-record batch — boundary case
single_as_batch = {"records": [SINGLE_REQUEST["features"]]}
r = client.post("/predict/batch", json=single_as_batch, headers=AUTH)
print(f"Single-record batch → {r.status_code} (expected 200)")
assert r.status_code == 200
assert len(r.json()["predictions"]) == 1

print("\n✅  /predict/batch edge cases OK")

## 7. Run `test_api.py` — Full Pytest Suite

Runs the complete `test_api.py` using pytest. Requires `artifacts/model.pkl`.

To isolate a specific test function, pass `-k <test_name>` — see §8.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure
python -m pytest tests/test_api.py -v --tb=short

## 8. Run Individual API Tests

Runs each API test group in sequence, printing a header and pass/fail per group.

**To run a single group:** comment out the others in the `MODULES` list below.
**To run a single test function:** set `EXTRA_ARGS = ["-k", "test_name"]` before running.

In [ ]:
import subprocess
import sys as _sys

# Each entry: (pytest -k expression, human-readable label)
GROUPS = [
    ("health",        "/health endpoint"),
    ("predict and not batch",  "/predict single-record"),
    ("batch",         "/predict/batch"),
]

group_results = {}
for k_expr, label in GROUPS:
    print(f"\n{'='*60}")
    print(f"  {label}  (-k '{k_expr}')")
    print(f"{'='*60}")
    r = subprocess.run(
        [_sys.executable, "-m", "pytest", "tests/test_api.py",
         "-v", "--tb=short", "-k", k_expr],
        cwd=ROOT,
    )
    group_results[label] = "✅ PASSED" if r.returncode == 0 else "❌ FAILED"

print(f"\n{'='*60}")
print("  Results Summary")
print(f"{'='*60}")
for lbl, status in group_results.items():
    print(f"  {status}  {lbl}")

## 9. Run `test_pipelines.py` — End-to-End Pipeline Tests

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure
python -m pytest tests/test_pipelines.py -v --tb=short

---

## Summary

| Test | Endpoint | Expected result |
|---|---|---|
| Health up | `GET /health` | `200 {"status": "healthy"}` |
| Health down | `GET /health` (no model) | `503 {"status": "unhealthy"}` |
| Single predict | `POST /predict` | `200 {prediction, probability, label}` |
| Empty features | `POST /predict` | `422` |
| No/wrong auth | `POST /predict` | `401` |
| Batch predict | `POST /predict/batch` | `200 {"predictions": [...]}` |
| Empty batch | `POST /predict/batch` | `422` |
| Batch >1000 | `POST /predict/batch` | `422` |
| pytest test_api | `tests/test_api.py` | All tests pass |
| pytest pipelines | `tests/test_pipelines.py` | All tests pass |

Once prediction tests pass, open **`04_docker_testing.ipynb`** to verify the same logic inside a Docker container.